# Insulin Resistance Prediction
**Dataset:** health_dataset_200k_real_countries.csv  
**Target:** Insulin_Resistant (0 = No, 1 = Yes)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, pickle
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

## 1. Load Data

In [ ]:
df = pd.read_csv('health_dataset_200k_real_countries.csv')
df.columns = df.columns.str.strip()
print(df.shape)
df.head()

## 2. EDA

In [ ]:
print(df.dtypes)
print("Missing values:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
# Target distribution
df['Insulin_Resistant'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'])
plt.title('Class Distribution')
plt.xlabel('Insulin Resistant')
plt.ylabel('Count')
plt.xticks([0, 1], ['No', 'Yes'], rotation=0)
plt.show()

In [ ]:
# Numerical feature distributions
num_cols = ['Age', 'HbA1c', 'HDL', 'LDL', 'TG']
df[num_cols].hist(figsize=(12, 8), bins=40)
plt.suptitle('Numerical Feature Distributions')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols + ['Insulin_Resistant']].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 3. Preprocessing

**Note on leaky features:** HOMA_IR, Fasting_Insulin, TG_HDL_Ratio, and Diabetic are removed because they either directly encode the target label or are near-perfect proxies for it, causing all models to score AUC = 1.0.

In [ ]:
from sklearn.preprocessing import LabelEncoder

ENCODE_COLS = ['Country', 'Gender', 'Age_Group', 'Heart_Risk']
df_enc = df.copy()
label_encoders = {}

for col in ENCODE_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    label_encoders[col] = le
    with open(f'label_encoder_{col}.pkl', 'wb') as f:
        pickle.dump(le, f)

print('Label encoding done')

In [ ]:
LEAKY_FEATURES = ['HOMA_IR', 'Fasting_Insulin', 'TG_HDL_Ratio', 'Diabetic']
X = df_enc.drop(columns=['ID', 'Insulin_Resistant'] + LEAKY_FEATURES)
y = df_enc['Insulin_Resistant']

print('Features:', X.columns.tolist())
print('Shape:', X.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]}  Test: {X_test.shape[0]}')

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## 4. SMOTE — Handle Class Imbalance

Applied only on training data. Test set is never touched.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print(f'Before SMOTE: {dict(y_train.value_counts())}')
print(f'After  SMOTE: {dict(pd.Series(y_train_sm).value_counts())}')

## 5. Model Training

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, f1_score

models = {
    'KNN':                 KNeighborsClassifier(n_neighbors=7),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                         eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0),
}

In [ ]:
# 5-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f'{"Model":<22} {"Accuracy":>10} {"F1":>8} {"ROC-AUC":>10}')
print('-' * 52)

for name, model in models.items():
    cv = cross_validate(model, X_train_sm, y_train_sm, cv=skf,
                        scoring=['accuracy', 'f1', 'roc_auc'])
    print(f'{name:<22} {cv["test_accuracy"].mean():>10.4f} {cv["test_f1"].mean():>8.4f} {cv["test_roc_auc"].mean():>10.4f}')

In [ ]:
# Train on full SMOTE training set, evaluate on original test set
print(f'{"Model":<22} {"Accuracy":>10} {"F1":>8} {"ROC-AUC":>10}')
print('-' * 52)

for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    print(f'{name:<22} {accuracy_score(y_test, y_pred):>10.4f} {f1_score(y_test, y_pred):>8.4f} {roc_auc_score(y_test, y_prob):>10.4f}')

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = list(axes.flat)

for i, (name, model) in enumerate(models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test_sc))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_flat[i],
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    axes_flat[i].set_title(name)
    axes_flat[i].set_xlabel('Predicted')
    axes_flat[i].set_ylabel('Actual')

axes_flat[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.show()

In [ ]:
# Feature Importances (Random Forest)
rf = models['Random Forest']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

feat_imp.plot(kind='bar', figsize=(10, 4), color='steelblue')
plt.title('Feature Importances - Random Forest')
plt.ylabel('Importance')
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 6. Save Best Model

In [ ]:
best_model = models['Random Forest']
with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('Saved model.pkl')